In this lab, you will compute the convergence score of the agent, and you will use Phoenix experiments to set up this evaluation. The video goes through the components of an experiment (dataset consisting of examples, task, evaluator). 

Here's how you will define the experiment components in this lab to compute the convergence score:

<img src="images/experiment1.png" width="700"/>

You will create a dataset of Phoenix examples where each example contains a different version of the same question. You will then create the task `run_agent_and_track_path` that you will use to run on each example and compute the path length. Finally, you will create the evaluator `evaluate_path_length` that computes the convergence score.

In [1]:
import phoenix as px
from phoenix.client.experiments import run_experiment, evaluate_experiment, create_evaluator
from phoenix.client.resources.datasets import DatasetExample
from phoenix.otel import register
import pandas as pd
from datetime import datetime
import os
import nest_asyncio
nest_asyncio.apply()

c:\Users\Nada\Documents\sales_agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_NAME = "trajectory-agent"
from openai import OpenAI
import pandas as pd
import json
import duckdb
from pydantic import BaseModel, Field
from IPython.display import Markdown


import os
from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry.trace import Status, StatusCode
from openinference.instrumentation import TracerProvider
from phoenix.otel import register
from helper import get_openai_api_key, get_phoenix_trace_endpoint

# initialize the OpenAI client
openai_api_key = get_openai_api_key()
client = OpenAI(api_key=openai_api_key)
MODEL = "gpt-4o-mini"

PROJECT_NAME = "evaluating-agent"
tracer_provider = register(
    project_name=PROJECT_NAME,
    endpoint=get_phoenix_trace_endpoint(),
    protocol="http/protobuf",
)

OpenAIInstrumentor().instrument(tracer_provider = tracer_provider)
tracer = tracer_provider.get_tracer(__name__)


# Defining the tools

# Tool 1: Database Lookup
TRANSACTION_DATA_FILE_PATH = './data/Store_Sales_Price_Elasticity_Promotions_Data.parquet'

# prompt template for step 2 of tool 1
SQL_GENERATION_PROMPT = """
Generate an SQL query based on a prompt. Do not reply with anything besides the SQL query.
The prompt is: {prompt}

The available columns are: {columns}
The table name is: {table_name}
"""

# code for step 2 of tool 1
def generate_sql_query(prompt: str, columns: list, table_name: str) -> str:
    """Generate an SQL query based on a prompt"""
    formatted_prompt = SQL_GENERATION_PROMPT.format(prompt=prompt, columns=columns, table_name=table_name)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    
    return response.choices[0].message.content


# code for tool 1
@tracer.tool()
def lookup_sales_data(prompt: str) -> str:
    """Implementation of sales data lookup from parquet file using SQL"""
    try:

        # define the table name
        table_name = "sales"
        
        # step 1: read the parquet file into a DuckDB table
        df = pd.read_parquet(TRANSACTION_DATA_FILE_PATH)
        duckdb.sql(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM df")

        # step 2: generate the SQL code
        sql_query = generate_sql_query(prompt, df.columns, table_name)
        # clean the response to make sure it only includes the SQL code
        sql_query = sql_query.strip()
        sql_query = sql_query.replace("```sql", "").replace("```", "")
        
        # step 3: execute the SQL query

        with tracer.start_as_current_span("execute_sql_query", openinference_span_kind="chain") as span:
            
            result = duckdb.sql(sql_query).df()

            span.set_output(value=str(result))
            span.set_status(StatusCode.OK)
        
        return result.to_string()
    except Exception as e:
        return f"Error accessing data: {str(e)}"


#### Tool 2: Data Analysis


# Construct prompt based on analysis type and data subset
DATA_ANALYSIS_PROMPT = """
Analyze the following data: {data}
Your job is to answer the following question: {prompt}
"""

# code for tool 2
@tracer.tool()
def analyze_sales_data(prompt: str, data: str) -> str:
    """Implementation of AI-powered sales data analysis"""
    formatted_prompt = DATA_ANALYSIS_PROMPT.format(data=data, prompt=prompt)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    
    analysis = response.choices[0].message.content
    return analysis if analysis else "No analysis could be generated"


# ### Tool 3: Data Visualization
    
# prompt template for step 1 of tool 3
CHART_CONFIGURATION_PROMPT = """
Generate a chart configuration based on this data: {data}
The goal is to show: {visualization_goal}
"""

# class defining the response format of step 1 of tool 3
class VisualizationConfig(BaseModel):
    chart_type: str = Field(..., description="Type of chart to generate")
    x_axis: str = Field(..., description="Name of the x-axis column")
    y_axis: str = Field(..., description="Name of the y-axis column")
    title: str = Field(..., description="Title of the chart")


# code for step 1 of tool 3
@tracer.chain()
def extract_chart_config(data: str, visualization_goal: str) -> dict:
    """Generate chart visualization configuration
    
    Args:
        data: String containing the data to visualize
        visualization_goal: Description of what the visualization should show
        
    Returns:
        Dictionary containing line chart configuration
    """
    formatted_prompt = CHART_CONFIGURATION_PROMPT.format(data=data, visualization_goal=visualization_goal)
    
    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
        response_format=VisualizationConfig,
    )
    
    try:
        # Extract axis and title info from response
        content = response.choices[0].message.content
        
        # Return structured chart config
        return {
            "chart_type": content.chart_type,
            "x_axis": content.x_axis,
            "y_axis": content.y_axis,
            "title": content.title,
            "data": data
        }
    except Exception:
        return {
            "chart_type": "line", 
            "x_axis": "date",
            "y_axis": "value",
            "title": visualization_goal,
            "data": data
        }



CREATE_CHART_PROMPT = """
Write python code to create a chart based on the following configuration.
Only return the code, no other text.
config: {config}
"""


# code for step 2 of tool 3
@tracer.chain()
def create_chart(config: dict) -> str:
    """Create a chart based on the configuration"""
    formatted_prompt = CREATE_CHART_PROMPT.format(config=config)
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    
    code = response.choices[0].message.content
    code = code.replace("```python", "").replace("```", "")
    code = code.strip()
    
    return code

    
# code for tool 3
@tracer.tool()
def generate_visualization(data: str, visualization_goal: str) -> str:
    """Generate a visualization based on the data and goal"""
    config = extract_chart_config(data, visualization_goal)
    code = create_chart(config)
    return code


# ## Tool Schema

    
# Define tools/functions that can be called by the model
tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_sales_data",
            "description": "Look up data from Store Sales Price Elasticity Promotions dataset",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {"type": "string", "description": "The unchanged prompt that the user provided."}
                },
                "required": ["prompt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_sales_data", 
            "description": "Analyze sales data to extract insights",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {"type": "string", "description": "The lookup_sales_data tool's output."},
                    "prompt": {"type": "string", "description": "The unchanged prompt that the user provided."}
                },
                "required": ["data", "prompt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_visualization",
            "description": "Generate Python code to create data visualizations",
            "parameters": {
                "type": "object", 
                "properties": {
                    "data": {"type": "string", "description": "The lookup_sales_data tool's output."},
                    "visualization_goal": {"type": "string", "description": "The goal of the visualization."}
                },
                "required": ["data", "visualization_goal"]
            }
        }
    }
]

# Dictionary mapping function names to their implementations
tool_implementations = {
    "lookup_sales_data": lookup_sales_data,
    "analyze_sales_data": analyze_sales_data, 
    "generate_visualization": generate_visualization
}



# code for executing the tools returned in the model's response
@tracer.chain()
def handle_tool_calls(tool_calls, messages):
    
    for tool_call in tool_calls:
        function = tool_implementations[tool_call.function.name]
        function_args = json.loads(tool_call.function.arguments)
        result = function(**function_args)
       
        messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
        
    return messages




SYSTEM_PROMPT = """
You are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.
"""



# The return of this method is different to match the required output dataset
def run_agent(messages):
    print("Running agent with messages:", messages)
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
        print("Converted string message to list format")
    
    # Check and add system prompt if needed
    if not any(
            isinstance(message, dict) and message.get("role") == "system" for message in messages
        ):
            system_prompt = {"role": "system", "content": "You are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset."}
            messages.append(system_prompt)
            print("Added system prompt to messages")

    while True:
        # Router call span
        print("Starting router")
            
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )

        messages.append(response.choices[0].message.model_dump())
        tool_calls = response.choices[0].message.tool_calls
        print("Received response with tool calls:", bool(tool_calls))
        
        if tool_calls:
            # Tool calls span
            print("Processing tool calls")
            tool_calls = response.choices[0].message.tool_calls
            messages = handle_tool_calls(tool_calls, messages)
        else:
            print("No tool calls, returning final response")
            return messages


### Creating the Main Span
def start_main_span(messages):
    print("Starting main span with messages:", messages)

    # This is manual tracing call.
    with tracer.start_as_current_span(
        "AgentRun", openinference_span_kind="agent"
    ) as span:
        span.set_input(value=messages)
        # Call the run_agent function to execute the agent's logic
        ret = run_agent(messages)
        print("Main span completed with return value:", ret)
        span.set_output(value=ret)
        span.set_status(StatusCode.OK)
        return ret


Phoenix endpoint: http://localhost:6006/
OpenTelemetry Tracing Details
|  Phoenix Project: evaluating-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [3]:
from phoenix.client import Client as px_client


In [4]:
# create dataset of testcases
# You can see this dataset in Phoenix dashboard under Datasets (outside of Tracing)
# Same queries different ways
convergence_questions = [
    "What was the average quantity sold per transaction?",
    "What is the mean number of items per sale?", 
    "Calculate the typical quantity per transaction",
    "What's the mean transaction size in terms of quantity?",
    "On average, how many items were purchased per transaction?",
    "What is the average basket size per sale?",
    "Calculate the mean number of products per purchase",
    "What's the typical number of units per order?",
    "What is the average number of products bought per purchase?",
    "Tell me the mean quantity of items in a typical transaction",
    "How many items does a customer buy on average per transaction?",
    "What's the usual number of units in each sale?",
    "What is the typical amount of products per transaction?",
    "Show the mean number of items customers purchase per visit",
    "What's the average quantity of units per shopping trip?",
    "How many products do customers typically buy in one transaction?",
    "What is the standard basket size in terms of quantity?"
]

convergence_df = pd.DataFrame({
    'question': convergence_questions
})

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
dataset = px_client().datasets.create_dataset(
    dataframe=convergence_df, 
    name=f"convergence_questions-{now}",
    input_keys=["question"])


On the created dataset we should experiments on the agent and compare their results.
Within each experiment there will be some tasks to run.
<img src="images/experiment1.png" width="700"/>

## Creating the Task

In [5]:
# helper method to format the output returned by the task
def format_message_steps(messages=None):
    """
    Convert a list of message objects into a readable format that shows the steps taken.

    Args:
        messages (list): A list of message objects containing role, content, tool calls, etc.

    Returns:
        str: A readable string showing the steps taken.
    """
    steps = []
    for message in messages:
        role = message.get("role")
        if role == "user":
            steps.append(f"User: {message.get('content')}")
        elif role == "system":
            steps.append("System: Provided context")
        elif role == "assistant":
            if message.get("tool_calls"):
                for tool_call in message["tool_calls"]:
                    tool_name = tool_call["function"]["name"]
                    steps.append(f"Assistant: Called tool '{tool_name}'")
            else:
                steps.append(f"Assistant: {message.get('content')}")
        elif role == "tool":
            steps.append(f"Tool response: {message.get('content')}")
    
    return "\n".join(steps)

In [6]:
def run_agent_and_track_path(example: DatasetExample) -> str:
    messages = [{"role": "user", "content": example.input.get("question")}]
    ret = run_agent(messages)
    print("rettt", ret)
    return {"path_length": len(ret), "messages": format_message_steps(ret)}

## Running the Experiment


In [ ]:
experiment = run_experiment(
    dataset=dataset,
    task=run_agent_and_track_path,
    experiment_name="Convergence Eval",
    experiment_description="Evaluating the convergence of the agent")

🧪 Experiment started.
📺 View dataset experiments: http://localhost:6006/datasets/RGF0YXNldDox/experiments
🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNldDox/compare?experimentId=RXhwZXJpbWVudDox


running tasks |          | 0/17 (0.0%) | ⏳ 00:00<? | ?it/s

Running agent with messages: [{'role': 'user', 'content': 'What was the average quantity sold per transaction?'}]
Added system prompt to messages
Starting router
Received response with tool calls: True
Processing tool calls
Starting router
Received response with tool calls: False
No tool calls, returning final response
rettt [{'role': 'user', 'content': 'What was the average quantity sold per transaction?'}, {'role': 'system', 'content': 'You are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.'}, {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': [], 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_uxUt9Qiekd20Tqv9h6gzTxJY', 'function': {'arguments': '{"prompt":"What was the average quantity sold per transaction?"}', 'name': 'lookup_sales_data'}, 'type': 'function'}]}, {'role': 'tool', 'content': '   Average_Quantity_Sold_Per_Transaction\n0                               1.484536', 'tool_call_i

running tasks |▌         | 1/17 (5.9%) | ⏳ 00:10<02:41 | 10.09s/it

Running agent with messages: [{'role': 'user', 'content': 'What is the mean number of items per sale?'}]
Added system prompt to messages
Starting router
Received response with tool calls: True
Processing tool calls
Starting router
Received response with tool calls: False
No tool calls, returning final response
rettt [{'role': 'user', 'content': 'What is the mean number of items per sale?'}, {'role': 'system', 'content': 'You are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.'}, {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': [], 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_7uDsN4DhmCxchcNloaHY54Bt', 'function': {'arguments': '{"prompt":"What is the mean number of items per sale?"}', 'name': 'lookup_sales_data'}, 'type': 'function'}]}, {'role': 'tool', 'content': '   mean_items_per_sale\n0             1.484536', 'tool_call_id': 'call_7uDsN4DhmCxchcNloaHY54Bt'}, {'content': 'The mean num

running tasks |█▏        | 2/17 (11.8%) | ⏳ 00:19<02:23 |  9.55s/it

... (9 outputs left out)

running tasks |████      | 7/17 (41.2%) | ⏳ 00:36<00:33 |  3.34s/it

Received response with tool calls: False
No tool calls, returning final response
rettt [{'role': 'user', 'content': 'Calculate the mean number of products per purchase'}, {'role': 'system', 'content': 'You are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.'}, {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': [], 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_yJmYPOEPJ10pV66b2EabOttM', 'function': {'arguments': '{"prompt":"Calculate the mean number of products per purchase"}', 'name': 'lookup_sales_data'}, 'type': 'function'}]}, {'role': 'tool', 'content': '   mean_products_per_purchase\n0                  799.420389', 'tool_call_id': 'call_yJmYPOEPJ10pV66b2EabOttM'}, {'content': 'The mean number of products per purchase is approximately 799.42.', 'refusal': None, 'role': 'assistant', 'annotations': [], 'audio': None, 'function_call': None, 'tool_calls': None}]
Running agent with messages: 

running tasks |████▋     | 8/17 (47.1%) | ⏳ 00:38<00:25 |  2.87s/it

Received response with tool calls: False
No tool calls, returning final response
rettt [{'role': 'user', 'content': "What's the typical number of units per order?"}, {'role': 'system', 'content': 'You are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.'}, {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': [], 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_7Uqgc9d1lpG9JCb9DlURhZzb', 'function': {'arguments': '{"prompt":"What\'s the typical number of units per order?"}', 'name': 'lookup_sales_data'}, 'type': 'function'}]}, {'role': 'tool', 'content': '   Average_Units_Per_Order\n0                 1.484536', 'tool_call_id': 'call_7Uqgc9d1lpG9JCb9DlURhZzb'}, {'content': 'The typical number of units per order is approximately 1.48 units.', 'refusal': None, 'role': 'assistant', 'annotations': [], 'audio': None, 'function_call': None, 'tool_calls': None}]
Running agent with messages: [{'role': 'use

In [ ]:
print(experiment)
# experiment_df = pd.DataFrame(experiment)


In [ ]:
# Get min optimal path length from the experiment results
# Notice that in experiment objects there is a list of task_runs, each containing the output of the task execution. We can extract the path_length from each run's output and find the minimum value.
min_optimal_path_length = min(
    run["output"]["path_length"]
    for run in experiment["task_runs"]
)
print(min_optimal_path_length)

Add evalutors to these experiments

In [ ]:

@create_evaluator(name="Convergence Eval", kind="CODE")
# decrator is optional but it is good bec it allows you to name it.
def evaluate_path_length(output: str) -> float:
    if output and output.get("path_length"):
        return min_optimal_path_length/float(output.get("path_length"))
    else:
        return 0

In [ ]:
experiment = evaluate_experiment(
    experiment=experiment,
    evaluators=[evaluate_path_length])